In [0]:
%run ./00_config_setup

#### 1. S3 bucket path

# Catalog

In [0]:
# Create all catalogs and schemas first
spark.sql(f"CREATE CATALOG  IF NOT EXISTS {CATALOG}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA   IF NOT EXISTS {BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA   IF NOT EXISTS {SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA   IF NOT EXISTS {DQ_SCHEMA}")
spark.sql(f"CREATE SCHEMA   IF NOT EXISTS {GOLD_SCHEMA}")
spark.sql(f"CREATE SCHEMA   IF NOT EXISTS {PLATINUM_SCHEMA}")

DataFrame[]

# Bronze

In [0]:
# bronze 
print("Bronze: tables created on first write inside 01_bronze_ingestion")

Bronze: tables created on first write inside 01_bronze_ingestion


# Silver

## 1.  silver_checkpoint

In [0]:
# SILVER

DELTA_PROPERTIES = """
    'delta.autoOptimize.optimizeWrite'    = 'true',
    'delta.autoOptimize.autoCompact'      = 'true',
    'delta.enableChangeDataFeed'          = 'true',
    'delta.logRetentionDuration'          = 'interval 30 days',
    'delta.deletedFileRetentionDuration'  = 'interval 7 days'
"""

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}.silver_checkpoint (
        batch_id            STRING,
        domain              STRING,
        processed_timestamp TIMESTAMP
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact'   = 'true'
    )
""")


DataFrame[]

## 2. electronics

In [0]:

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TBL_SLV_ELECTRONICS} (
        order_id                  STRING,
        product                   STRING,
        quantity_ordered          INT,
        quantity_ordered_raw      STRING,
        price_each                DOUBLE,
        price_each_raw            STRING,
        purchase_address          STRING,
        source_month              STRING,
        order_timestamp           TIMESTAMP,
        order_date_dt             DATE,
        order_year                INT,
        order_month               INT,
        order_quarter             INT,
        order_hour                INT,
        is_weekend                BOOLEAN,
        is_cross_year             BOOLEAN,
        street                    STRING,
        city                      STRING,
        state                     STRING,
        zip_code                  STRING,
        line_revenue              DOUBLE,
        order_id_count            INT,
        is_duplicate_order_id     BOOLEAN,
        has_valid_quantity         BOOLEAN,
        has_valid_price            BOOLEAN,
        has_valid_timestamp        BOOLEAN,
        has_valid_address          BOOLEAN,
        row_quality_flag           STRING,
        ingestion_timestamp        TIMESTAMP,
        ingestion_date             DATE,
        batch_id                   STRING,
        source_file_name           STRING,
        domain                     STRING,
        silver_timestamp           TIMESTAMP,
        silver_batch_id            STRING,
        silver_domain              STRING
    )
    USING DELTA
    LOCATION '{S3_SILVER_ELECTRONICS}'
    PARTITIONED BY (order_year, order_month)
    TBLPROPERTIES ({DELTA_PROPERTIES})
""")


DataFrame[]

## 3. Liquor

In [0]:

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TBL_SLV_LIQUOR} (
        invoice_item_number       STRING,
        date                      STRING,
        store_number              INT,
        store_name                STRING,
        address                   STRING,
        city                      STRING,
        zip_code                  STRING,
        store_location            STRING,
        county_number             INT,
        county                    STRING,
        category                  STRING,
        category_name             STRING,
        vendor_number             INT,
        vendor_name               STRING,
        item_number               INT,
        item_description          STRING,
        pack                      INT,
        bottle_volume__ml_        INT,
        state_bottle_cost         DOUBLE,
        state_bottle_retail       DOUBLE,
        bottles_sold              INT,
        sale__dollars_            DOUBLE,
        volume_sold__liters_      DOUBLE,
        volume_sold__gallons_     DOUBLE,
        transaction_date          DATE,
        transaction_year          INT,
        transaction_month         INT,
        transaction_quarter       INT,
        lat                       DOUBLE,
        lon                       DOUBLE,
        has_geo                   BOOLEAN,
        geo_quality               STRING,
        sale_matches_calc         BOOLEAN,
        row_quality_flag          STRING,
        ingestion_timestamp       TIMESTAMP,
        ingestion_date            DATE,
        batch_id                  STRING,
        source_file_name          STRING,
        domain                    STRING,
        silver_timestamp          TIMESTAMP,
        silver_batch_id           STRING,
        silver_domain             STRING
    )
    USING DELTA
    LOCATION '{S3_SILVER_LIQUOR}'
    PARTITIONED BY (transaction_year, transaction_month)
    TBLPROPERTIES ({DELTA_PROPERTIES})
""")


DataFrame[]

## 4. Books data

In [0]:

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TBL_SLV_BOOKS_DATA} (
        title                     STRING,
        authors                   STRING,
        description               STRING,
        publisher                 STRING,
        publisheddate             STRING,
        categories                STRING,
        ratingscount              STRING,
        primary_author            STRING,
        primary_category          STRING,
        published_date            DATE,
        published_year            INT,
        year_plausibility_flag    STRING,
        ratings_count             INT,
        has_description           BOOLEAN,
        has_author                BOOLEAN,
        has_publisher             BOOLEAN,
        row_quality_flag          STRING,
        ingestion_timestamp       TIMESTAMP,
        ingestion_date            DATE,
        batch_id                  STRING,
        source_file_name          STRING,
        domain                    STRING,
        silver_timestamp          TIMESTAMP,
        silver_batch_id           STRING,
        silver_domain             STRING
    )
    USING DELTA
    LOCATION '{S3_SILVER_BOOKS_DATA}'
    TBLPROPERTIES ({DELTA_PROPERTIES})
""")



DataFrame[]

## 5. Book_ratings

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TBL_SLV_BOOKS_RATING} (
        review_id                  STRING,
        book_id                    STRING,
        title                      STRING,
        price                      DOUBLE,
        user_id                    STRING,
        profilename                STRING,
        review_helpfulness         STRING,
        helpful_votes              INT,
        total_votes                INT,
        helpfulness_ratio          DOUBLE,
        review_score               DOUBLE,
        is_valid_score             BOOLEAN,
        review_time                STRING,
        review_date                DATE,
        review_year                INT,
        review_month               INT,
        is_valid_review_date       BOOLEAN,
        review_summary             STRING,
        review_text                STRING,
        review_text_truncated      STRING,
        review_text_was_truncated  BOOLEAN,
        row_quality_flag           STRING,
        ingestion_timestamp        TIMESTAMP,
        ingestion_date             DATE,
        batch_id                   STRING,
        source_file_name           STRING,
        domain                     STRING,
        silver_timestamp           TIMESTAMP,
        silver_batch_id            STRING,
        silver_domain              STRING
    )
    USING DELTA
    LOCATION '{S3_SILVER_BOOKS_RATING}'
    PARTITIONED BY (review_year)
    TBLPROPERTIES ({DELTA_PROPERTIES})
""")


DataFrame[]

# Dq

## 1. dq_logs

In [0]:
# DQ LAYER

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TBL_DQ_LOGS} (
        log_id          STRING    COMMENT 'UUID for this log row',
        run_id          STRING    COMMENT 'Groups all tests in one DQ run',
        run_timestamp   TIMESTAMP COMMENT 'UTC time this test ran',
        run_date        STRING    COMMENT 'YYYY-MM-DD',
        domain          STRING    COMMENT 'electronics / liquor / books_data / books_rating',
        table_name      STRING    COMMENT 'Fully qualified Silver table',
        test_name       STRING    COMMENT 'Short test identifier',
        test_category   STRING    COMMENT 'completeness / uniqueness / accuracy / range / timeliness / consistency',
        rows_checked    LONG      COMMENT 'Total rows in Silver table',
        rows_failed     LONG      COMMENT 'Rows that violated this rule',
        threshold_pct   DOUBLE    COMMENT 'Minimum pass rate — NULL means must be zero failures',
        pass_rate_pct   DOUBLE    COMMENT '(rows_checked - rows_failed) / rows_checked * 100',
        status          STRING    COMMENT 'PASS / WARN / FAIL',
        notes           STRING    COMMENT 'Explanation'
    )
    USING DELTA
    LOCATION '{S3_DQ_LOGS}'
    TBLPROPERTIES (
        'delta.logRetentionDuration'         = 'interval 30 days',
        'delta.deletedFileRetentionDuration' = 'interval 7 days'
    )
    COMMENT 'DQ test results'
""")


DataFrame[]

## 2. dq_checkpoints

In [0]:

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS retailflow360.{DQ_SCHEMA}.dq_checkpoint (
        domain       STRING,
        batch_id     STRING,
        processed_at TIMESTAMP
    )
    USING DELTA
""")


DataFrame[]

## 3. quarantine_tables


In [0]:

for tbl, s3 in [
    (TBL_Q_ELEC, S3_Q_ELEC),
    (TBL_Q_LIQ,  S3_Q_LIQ),
    (TBL_Q_BD,   S3_Q_BD),
    (TBL_Q_BR,   S3_Q_BR),
]:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tbl} (
            dq_fail_reasons   STRING    COMMENT 'Which DQ rules this row violated',
            quarantine_run_id STRING    COMMENT 'DQ run that identified this row',
            quarantine_date   STRING    COMMENT 'YYYY-MM-DD'
        )
        USING DELTA
        LOCATION '{s3}'
        TBLPROPERTIES (
            'delta.columnMapping.mode' = 'name',
            'delta.minReaderVersion'   = '2',
            'delta.minWriterVersion'   = '5'
        )
    """)

# Gold

## 1. Dimension_tables

In [0]:
# GOLD LAYER

GOLD_TBLPROPERTIES = """
    'delta.autoOptimize.optimizeWrite'   = 'true',
    'delta.autoOptimize.autoCompact'     = 'true',
    'delta.enableChangeDataFeed'         = 'true',
    'delta.logRetentionDuration'         = 'interval 30 days',
    'delta.deletedFileRetentionDuration' = 'interval 7 days'
"""

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TBL_DIM_DATE} (
        date_key     INT,
        full_date    DATE,
        year         INT,
        month        INT,
        month_name   STRING,
        quarter      INT,
        quarter_name STRING,
        week_of_year INT,
        day_of_week  INT,
        day_name     STRING,
        is_weekend   BOOLEAN,
        is_month_end BOOLEAN
    )
    USING DELTA
    LOCATION '{S3_GOLD_DIM_DATE}'
    TBLPROPERTIES ({GOLD_TBLPROPERTIES})
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TBL_DIM_DOMAIN} (
        domain_key  STRING,
        domain_name STRING,
        domain_desc STRING,
        created_at  TIMESTAMP
    )
    USING DELTA
    LOCATION '{S3_GOLD_DIM_DOMAIN}'
    TBLPROPERTIES ({GOLD_TBLPROPERTIES})
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TBL_DIM_PRODUCT} (
        product_key  STRING,
        product_name STRING,
        category     STRING,
        domain       STRING,
        unit_price   DOUBLE,
        updated_at   TIMESTAMP
    )
    USING DELTA
    LOCATION '{S3_GOLD_DIM_PRODUCT}'
    TBLPROPERTIES ({GOLD_TBLPROPERTIES})
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TBL_DIM_GEOGRAPHY} (
        geo_key    STRING,
        city       STRING,
        state      STRING,
        zip_code   STRING,
        county     STRING,
        lat        DOUBLE,
        lon        DOUBLE,
        domain     STRING,
        updated_at TIMESTAMP
    )
    USING DELTA
    LOCATION '{S3_GOLD_DIM_GEOGRAPHY}'
    TBLPROPERTIES ({GOLD_TBLPROPERTIES})
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TBL_DIM_STORE} (
        store_key      STRING,
        store_number   INT,
        store_name     STRING,
        address        STRING,
        city           STRING,
        zip_code       STRING,
        county         STRING,
        effective_date DATE,
        expiry_date    DATE,
        is_current     BOOLEAN,
        created_at     TIMESTAMP
    )
    USING DELTA
    LOCATION '{S3_GOLD_DIM_STORE}'
    TBLPROPERTIES ({GOLD_TBLPROPERTIES})
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TBL_DIM_CUSTOMER} (
        customer_key   STRING,
        user_id        STRING,
        profile_name   STRING,
        is_anonymous   BOOLEAN,
        effective_date DATE,
        expiry_date    DATE,
        is_current     BOOLEAN,
        created_at     TIMESTAMP
    )
    USING DELTA
    LOCATION '{S3_GOLD_DIM_CUSTOMER}'
    TBLPROPERTIES ({GOLD_TBLPROPERTIES})
""")


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4887688419577447>, line 28
      2     return (
      3         builder
      4         .property("delta.autoOptimize.optimizeWrite",   "true")
   (...)
      8         .property("delta.deletedFileRetentionDuration", "interval 7 days")
      9     )
     11 # DIM_DATE
     12 add_gold_properties(
     13     DeltaTable.createIfNotExists(spark)
     14     .tableName(TBL_DIM_DATE)
     15     .location(S3_GOLD_DIM_DATE)
     16     .addColumn("date_key",     IntegerType())
     17     .addColumn("full_date",    DateType())
     18     .addColumn("year",         IntegerType())
     19     .addColumn("month",        IntegerType())
     20     .addColumn("month_name",   StringType())
     21     .addColumn("quarter",      IntegerType())
     22     .addColumn("quarter_name", StringType())
     23     .addColumn("week_of_year",

## 2. fact_table

In [0]:

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TBL_FACT_SALES} (
        sales_key        STRING,
        product_key      STRING,
        store_key        STRING,
        customer_key     STRING,
        date_key         INT,
        geo_key          STRING,
        domain_key       STRING,
        domain           STRING,
        quantity         INT,
        unit_price       DOUBLE,
        line_revenue     DOUBLE,
        review_score     DOUBLE,
        helpful_votes    INT,
        total_votes      INT,
        source_file_name STRING,
        batch_id         STRING,
        gold_timestamp   TIMESTAMP
    )
    USING DELTA
    LOCATION '{S3_GOLD_FACT_SALES}'
    PARTITIONED BY (domain)
    TBLPROPERTIES ({GOLD_TBLPROPERTIES})
""")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4887688419577449>, line 2
      1 # FACT_SALES
----> 2 add_gold_properties(
      3     DeltaTable.createIfNotExists(spark)
      4     .tableName(TBL_FACT_SALES)
      5     .location(S3_GOLD_FACT_SALES)
      6     .addColumn("sales_key",        StringType())
      7     .addColumn("product_key",      StringType())
      8     .addColumn("store_key",        StringType())
      9     .addColumn("customer_key",     StringType())
     10     .addColumn("date_key",         IntegerType())
     11     .addColumn("geo_key",          StringType())
     12     .addColumn("domain_key",       StringType())
     13     .addColumn("domain",           StringType())
     14     .addColumn("quantity",         IntegerType())
     15     .addColumn("unit_price",       DoubleType())
     16     .addColumn("line_revenue",     DoubleType())
